# Bài 4.1: Tinh chỉnh Nhẹ (QLoRA) Mô hình Ngôn ngữ Lớn (Qwen / Sailor2) cho Tiếng Việt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vietnamese-nlp-mastery/vietnamese-nlp-mastery/blob/main/04_Vietnamese_LLMs_and_Agents/01_qlora_finetuning_qwen_vietnamese.ipynb)

> **Mục tiêu bài học:**
> 1. Tổng quan các họ mô hình ngôn ngữ lớn (LLM) mã nguồn mở tốt nhất cho tiếng Việt hiện nay: **`Qwen2.5-7B-Instruct`**, **`Sailor2-8B-Chat`** (SeaAI Lab), và **`ura-hcmut/ura-llama`** (ĐH Bách Khoa TP.HCM).
> 2. Thực hành **Lượng tử hóa 4-bit NormalFloat (`NF4`)** và gắn Adapter **LoRA (`r=16, lora_alpha=32`)** lên các lớp Linear (`q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`).
> 3. Định dạng dữ liệu hội thoại chuẩn **ChatML Tiếng Việt** và huấn luyện với `SFTTrainer` (`trl` / `Unsloth`) chỉ với ~8-12GB VRAM.

In [1]:
# Minh họa định dạng dữ liệu Instruction Tuning theo chuẩn ChatML cho Qwen / Sailor2
sample_conversation = [
    {
        "role": "system",
        "content": "Bạn là trợ lý AI chuyên gia về xử lý ngôn ngữ tự nhiên tiếng Việt, trả lời chính xác và ngắn gọn.",
    },
    {
        "role": "user",
        "content": "Tại sao khi dùng PhoBERT lại cần tách từ bằng VnCoreNLP còn Qwen thì không cần?",
    },
    {
        "role": "assistant",
        "content": (
            "PhoBERT được tiền huấn luyện trên kho văn bản tiếng Việt đã tách từ ghép bằng RDRSegmenter "
            "(nối bằng dấu gạch dưới '_', ví dụ 'học_sinh'). Ngược lại, Qwen sử dụng Byte-level BPE Tokenizer "
            "huấn luyện trực tiếp trên văn bản tiếng Việt tự nhiên không nối gạch dưới."
        ),
    },
]

for turn in sample_conversation:
    print(f"<|im_start|>{turn['role']}\n{turn['content']}<|im_end|>")

<|im_start|>system
Bạn là trợ lý AI chuyên gia về xử lý ngôn ngữ tự nhiên tiếng Việt, trả lời chính xác và ngắn gọn.<|im_end|>
<|im_start|>user
Tại sao khi dùng PhoBERT lại cần tách từ bằng VnCoreNLP còn Qwen thì không cần?<|im_end|>
<|im_start|>assistant
PhoBERT được tiền huấn luyện trên kho văn bản tiếng Việt đã tách từ ghép bằng RDRSegmenter (nối bằng dấu gạch dưới '_', ví dụ 'học_sinh'). Ngược lại, Qwen sử dụng Byte-level BPE Tokenizer huấn luyện trực tiếp trên văn bản tiếng Việt tự nhiên không nối gạch dưới.<|im_end|>


## Cấu hình QLoRA 4-bit với `peft`, `bitsandbytes` và `trl` (`SFTTrainer`)

```python
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # Hoặc "sail/Sailor2-8B-Chat"

# 1. Cấu hình Lượng tử hóa 4-bit NF4 + Double Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

# 2. Cấu hình LoRA Adapter cho toàn bộ lớp Attention & MLP
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 3. Huấn luyện với SFTTrainer
sft_config = SFTConfig(
    output_dir="./outputs/qwen_vn_qlora",
    max_seq_length=2048,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    bf16=True,
    logging_steps=10,
)
```